In [4]:
import sys
import time
import random
import math
import copy
import heapq

# --- 1. 改善版 焼きなまし法 (ft10用) ---


# --- 2. 大規模問題用 ディスパッチング・ルール ---

class LargeScaleSolver:
    def __init__(self):
        # 規模設定: 500ジョブ x 200機械 = 100,000オペレーション
        self.n_jobs = 500
        self.n_machines = 200
        self.jobs_data = []

    def generate_massive_data(self):
        print(f"=== 2. 大規模問題 ({self.n_jobs * self.n_machines} Operations) ===")
        print("データを生成中...")
        random.seed(42)
        self.jobs_data = []
        for j in range(self.n_jobs):
            # 各ジョブはランダムな機械順序を通る
            machines = list(range(self.n_machines))
            random.shuffle(machines)
            ops = []
            for m in machines:
                duration = random.randint(1, 100)
                ops.append((m, duration))
            self.jobs_data.append(ops)
        print("データ生成完了。")

    def solve_by_dispatching_rule(self):
        """
        MWKR (Most Work Remaining) ルールに基づくグリーディ法
        """
        print("ディスパッチング・ルール(MWKR)で求解中...")
        start_time = time.time()

        # 各機械がいつ空くか
        machine_available_time = [0] * self.n_machines
        # 各ジョブの次の工程インデックス
        job_next_op_index = [0] * self.n_jobs
        # 各ジョブがいつ空くか（前の工程が終わる時間）
        job_available_time = [0] * self.n_jobs
        # 各ジョブの残り作業時間合計（MWKR用）
        job_remaining_work = []

        for j in range(self.n_jobs):
            total_time = sum(op[1] for op in self.jobs_data[j])
            job_remaining_work.append(total_time)

        # イベント管理用の優先度付きキュー
        # (available_time, job_id)
        # 実際は機械ごとに待ち行列を持つのが一般的だが、
        # ここでは簡易的に「全ジョブの中で着手可能なもの」を探すシミュレーションを行う

        # 高速化のため、機械ごとの待ち行列を管理
        machine_queues = [[] for _ in range(self.n_machines)]

        # 全ジョブの最初の工程をキューに入れる
        for j in range(self.n_jobs):
            first_op_machine = self.jobs_data[j][0][0]
            # 優先度: 残り作業時間が多い順 (負の値にして最小ヒープを使う)
            priority = -job_remaining_work[j]
            heapq.heappush(machine_queues[first_op_machine], (priority, j))

        completed_ops = 0
        total_ops = self.n_jobs * self.n_machines

        # 簡易イベントシミュレーション
        # ここでは正確な離散イベントシミュレーション(DES)ではなく、
        # 「機械が空いたらキューから取る」ロジックで近似的に計算

        # 機械が空く時間のイベントキュー (time, machine_id)
        events = []
        for m in range(self.n_machines):
            heapq.heappush(events, (0, m))

        while completed_ops < total_ops:
            # 最も早く空く機械を取り出す
            current_time, m_id = heapq.heappop(events)

            # その機械のキューに処理可能なジョブがあるか？
            # ジョブが到着している(job_available_time <= current_time)必要がある

            # 注: 本来は厳密なDESが必要だが、ここでは「機械が空いた時点で、
            # その機械待ちのジョブの中で、すでに到着しているものの中から優先度高いものを選ぶ」
            # というロジックにする。

            # キュー内の候補を探す（到着しているものだけ取り出したい）
            # 簡易化のため、キューの先頭を見る

            if not machine_queues[m_id]:
                # 待機ジョブなし。この機械は実質「待機中」。
                # 時間を進めるために、未来のイベントとして再登録したいが、
                # 単純化のため、ジョブの到着時刻までジャンプさせるロジックが必要。
                # 今回は大規模デモのため、機械の空き時間を更新せず、
                # 全機械・全ジョブを走査する簡易グリーディ法に切り替えます。
                break

        # --- シンプルかつ高速なグリーディ法（再実装） ---
        # 機械ごとの「次に処理するジョブ」を決定していく

        machine_end_time = [0] * self.n_machines
        job_end_time = [0] * self.n_jobs
        op_idx = [0] * self.n_jobs

        # まだ処理していないオペレーションがあるジョブのリスト
        active_jobs = list(range(self.n_jobs))

        while active_jobs:
            # ここでは「次に処理可能なオペレーション」の中で
            # 「最も早く開始できるもの」かつ「残り作業が多いもの」を選ぶ

            # 高速化: 単純に全ジョブを回す（10万回ループ x ステップ数）
            # これはPythonでは遅いので、論理的な解説に留めるレベルですが、
            # 実際に動く軽量ロジック：

            # 「各ジョブの次の工程」について、
            # start_time = max(job_end, machine_end) が最小になるものを選ぶ

            best_job = -1
            best_start = float('inf')

            # ランダムに100個サンプリングして決める（近似解法）
            # 10万オペレーションを真面目にやるとPythonでは分単位かかるため
            candidates = active_jobs if len(active_jobs) < 100 else random.sample(active_jobs, 100)

            for j in candidates:
                idx = op_idx[j]
                m_id, duration = self.jobs_data[j][idx]

                # 開始可能時刻
                s_time = max(job_end_time[j], machine_end_time[m_id])

                if s_time < best_start:
                    best_start = s_time
                    best_job = j
                elif s_time == best_start:
                    # タイブレーク: MWKR
                    if job_remaining_work[j] > job_remaining_work[best_job]:
                        best_job = j

            # 実行
            j = best_job
            idx = op_idx[j]
            m_id, duration = self.jobs_data[j][idx]

            end_time = best_start + duration
            job_end_time[j] = end_time
            machine_end_time[m_id] = end_time

            # 更新
            job_remaining_work[j] -= duration
            op_idx[j] += 1
            if op_idx[j] >= self.n_machines:
                active_jobs.remove(j)

        makespan = max(machine_end_time)
        elapsed = time.time() - start_time
        print(f"計算終了。所要時間: {elapsed:.2f}秒")
        print(f"Makespan: {makespan}")
        print(f"処理したオペレーション数: {self.n_jobs * self.n_machines}")


if __name__ == "__main__":
    # 1. ft10 改善版
    #sa = ImprovedSASolver()
    #sa.solve_ft10()

    # 2. 大規模問題
    ls = LargeScaleSolver()
    ls.generate_massive_data()
    ls.solve_by_dispatching_rule()


=== 2. 大規模問題 (100000 Operations) ===
データを生成中...
データ生成完了。
ディスパッチング・ルール(MWKR)で求解中...
計算終了。所要時間: 7.00秒
Makespan: 33025
処理したオペレーション数: 100000
